# 01. 데이터 적재

원자료 CSV를 읽어서 기본 점검(행 수, 기간, 중복)을 하고 PostgreSQL에 원본 그대로 넣는 노트북입니다.
커피만 골라 넣지 않고 전 업종을 넣습니다. 거르는 건 `sql/01_분석_테이블.sql`에서 합니다.

- 넣는 테이블: 원본_추정매출, 원본_점포, 원본_길단위인구, 원본_직장인구, 원본_상주인구, 원본_집객시설, 원본_영역
- 임대료 파일은 5단계에서 후보 3곳만 직접 맞춰 볼 거라서 여기서는 안 넣습니다.

In [1]:
import os
import glob
import unicodedata

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 0. DB 연결

접속 정보는 `.env`에 따로 적어 뒀습니다. `.env`는 `.gitignore`에 들어 있어서 깃허브에는 안 올라갑니다.

In [2]:
load_dotenv('../.env')

url = URL.create(
    drivername='postgresql+psycopg2',
    username=os.environ['PGUSER'],
    password=os.environ['PGPASSWORD'],
    host=os.environ['PGHOST'],
    port=int(os.environ['PGPORT']),
    database=os.environ['PGDATABASE'],
)
engine = create_engine(url)

# 연결 잘 되는지 한 번 찔러 보기
with engine.connect() as conn:
    print(conn.execute(text('SELECT current_database()')).scalar())

cafe_location_project


## 1. 파일 목록

맥에서는 한글 파일 이름이 자음/모음이 분리된 형태(NFD)로 저장돼서 `'집객시설' in 파일명` 같은 비교가 안 맞을 때가 있습니다.
그래서 파일 이름을 합쳐진 형태(NFC)로 바꿔서 딕셔너리에 넣어 두고, 이름으로 꺼내 쓰기로 했습니다.

In [3]:
RAW_DIR = '../data/raw'

files = {}
for path in sorted(glob.glob(RAW_DIR + '/*.csv')):
    name = unicodedata.normalize('NFC', os.path.basename(path))
    files[name] = path

for name in files:
    print(name)

서울시 상권분석서비스(길단위인구-상권).csv
서울시 상권분석서비스(상주인구-상권).csv
서울시 상권분석서비스(직장인구-상권).csv
서울시 상권분석서비스(집객시설-상권).csv
서울시_상권분석서비스(영역-상권).csv
서울시_상권분석서비스(점포-상권)_2021년.csv
서울시_상권분석서비스(점포-상권)_2022년.csv
서울시_상권분석서비스(점포-상권)_2023년.csv
서울시_상권분석서비스(점포-상권)_2024년.csv
서울시_상권분석서비스(점포-상권)_2025~2026년.csv
서울시_상권분석서비스(추정매출-상권)_2021년.csv
서울시_상권분석서비스(추정매출-상권)_2022년.csv
서울시_상권분석서비스(추정매출-상권)_2023년.csv
서울시_상권분석서비스(추정매출-상권)_2024년.csv
서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv
임대동향 지역별 임대료(2026년2분기)_소규모 상가.csv


## 2. 추정매출 (연도별 5개 파일)

5개 파일 컬럼이 같은지 먼저 보고 하나로 합칩니다. 인코딩은 전부 cp949입니다.

In [4]:
sales_files = [
    '서울시_상권분석서비스(추정매출-상권)_2021년.csv',
    '서울시_상권분석서비스(추정매출-상권)_2022년.csv',
    '서울시_상권분석서비스(추정매출-상권)_2023년.csv',
    '서울시_상권분석서비스(추정매출-상권)_2024년.csv',
    '서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv',
]

sales_list = []
for name in sales_files:
    df = pd.read_csv(files[name], encoding='cp949')
    print(name, df.shape)
    sales_list.append(df)

# 첫 파일이랑 컬럼이 다른 파일이 있는지 확인
first_cols = list(sales_list[0].columns)
for name, df in zip(sales_files, sales_list):
    print(name, '컬럼 같음' if list(df.columns) == first_cols else '컬럼 다름!')

서울시_상권분석서비스(추정매출-상권)_2021년.csv (89150, 55)


서울시_상권분석서비스(추정매출-상권)_2022년.csv (88834, 55)


서울시_상권분석서비스(추정매출-상권)_2023년.csv (88246, 55)


서울시_상권분석서비스(추정매출-상권)_2024년.csv (87179, 55)


서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv (128053, 55)
서울시_상권분석서비스(추정매출-상권)_2021년.csv 컬럼 같음
서울시_상권분석서비스(추정매출-상권)_2022년.csv 컬럼 같음
서울시_상권분석서비스(추정매출-상권)_2023년.csv 컬럼 같음
서울시_상권분석서비스(추정매출-상권)_2024년.csv 컬럼 같음
서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv 컬럼 같음


In [5]:
sales = pd.concat(sales_list, ignore_index=True)
print(sales.shape)
sales.head(3)

(481462, 55)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,...,시간대_건수~21_매출_건수,시간대_건수~24_매출_건수,남성_매출_건수,여성_매출_건수,연령대_10_매출_건수,연령대_20_매출_건수,연령대_30_매출_건수,연령대_40_매출_건수,연령대_50_매출_건수,연령대_60_이상_매출_건수
0,20211,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,382839289,13701,235308851,...,4520,141,8000,4228,102,826,1174,2467,4049,3615
1,20211,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,122957138,12039,61070094,...,552,0,6679,4293,52,926,1222,2698,3404,2670
2,20211,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,19012827,264,14322928,...,188,58,182,45,0,0,9,18,102,97


시간대 컬럼 이름에 `~`가 들어가 있어서 SQL에서 쓸 때마다 따옴표로 감싸야 합니다. 유동인구 파일처럼 `_`로 바꿨습니다.
건수 쪽은 원본 이름이 `시간대_건수~06_매출_건수`처럼 깨져 있어서 같이 고쳤습니다.

In [6]:
rename_map = {
    '시간대_00~06_매출_금액': '시간대_00_06_매출_금액',
    '시간대_06~11_매출_금액': '시간대_06_11_매출_금액',
    '시간대_11~14_매출_금액': '시간대_11_14_매출_금액',
    '시간대_14~17_매출_금액': '시간대_14_17_매출_금액',
    '시간대_17~21_매출_금액': '시간대_17_21_매출_금액',
    '시간대_21~24_매출_금액': '시간대_21_24_매출_금액',
    '시간대_건수~06_매출_건수': '시간대_00_06_매출_건수',
    '시간대_건수~11_매출_건수': '시간대_06_11_매출_건수',
    '시간대_건수~14_매출_건수': '시간대_11_14_매출_건수',
    '시간대_건수~17_매출_건수': '시간대_14_17_매출_건수',
    '시간대_건수~21_매출_건수': '시간대_17_21_매출_건수',
    '시간대_건수~24_매출_건수': '시간대_21_24_매출_건수',
}
sales = sales.rename(columns=rename_map)

# 이름에 ~ 남은 컬럼 있으면 여기서 보임
print([c for c in sales.columns if '~' in c])

[]


In [7]:
# 분기가 빠짐없이 들어왔는지 (2021Q1 ~ 2026Q2 = 22개)
print('분기 수:', sales['기준_년분기_코드'].nunique())
print(sales['기준_년분기_코드'].value_counts().sort_index())

분기 수: 22
기준_년분기_코드
20211    22239
20212    22325
20213    22232
20214    22354
20221    22255
20222    22250
20223    22113
20224    22216
20231    22106
20232    22083
20233    22017
20234    22040
20241    21910
20242    21887
20243    21718
20244    21664
20251    21427
20252    21452
20253    21520
20254    21333
20261    21188
20262    21133
Name: count, dtype: int64


In [8]:
# 같은 분기 + 상권 + 업종이 두 번 들어간 행이 있으면 안 됨
dup = sales.duplicated(['기준_년분기_코드', '상권_코드', '서비스_업종_코드'])
print('중복 행:', dup.sum())

# 결측
print('결측 있는 컬럼:', sales.columns[sales.isna().any()].tolist())

중복 행: 0
결측 있는 컬럼: []


In [9]:
# 커피-음료가 분기마다 몇 개 상권에 있는지만 미리 봐 둠
coffee_sales = sales[sales['서비스_업종_코드_명'] == '커피-음료']
coffee_sales.groupby('기준_년분기_코드')['상권_코드'].nunique()

기준_년분기_코드
20211    1047
20212    1057
20213    1067
20214    1072
20221    1070
20222    1073
20223    1083
20224    1092
20231    1085
20232    1097
20233    1092
20234    1093
20241    1087
20242    1084
20243    1088
20244    1089
20251    1077
20252    1073
20253    1077
20254    1069
20261    1064
20262    1064
Name: 상권_코드, dtype: int64

## 3. 점포 (연도별 5개 파일)

2025~2026년 파일만 컬럼 구성이 다릅니다. 일단 원본 그대로 합치고(없는 컬럼은 빈 값), 점포 수를 하나로 맞추는 건 SQL에서 합니다.

In [10]:
store_files = [
    '서울시_상권분석서비스(점포-상권)_2021년.csv',
    '서울시_상권분석서비스(점포-상권)_2022년.csv',
    '서울시_상권분석서비스(점포-상권)_2023년.csv',
    '서울시_상권분석서비스(점포-상권)_2024년.csv',
    '서울시_상권분석서비스(점포-상권)_2025~2026년.csv',
]

store_list = []
for name in store_files:
    df = pd.read_csv(files[name], encoding='cp949')
    print(name, df.shape)
    store_list.append(df)

서울시_상권분석서비스(점포-상권)_2021년.csv (303880, 14)


서울시_상권분석서비스(점포-상권)_2022년.csv (305587, 14)


서울시_상권분석서비스(점포-상권)_2023년.csv (307741, 14)


서울시_상권분석서비스(점포-상권)_2024년.csv (306889, 14)


서울시_상권분석서비스(점포-상권)_2025~2026년.csv (456659, 14)


In [11]:
# 2024년 파일과 2025~2026년 파일의 컬럼 차이
old_cols = set(store_list[3].columns)
new_cols = set(store_list[4].columns)
print('2024년에만 있는 컬럼:', old_cols - new_cols)
print('2025년부터 생긴 컬럼:', new_cols - old_cols)

2024년에만 있는 컬럼: {'점포_수', '유사_업종_점포_수'}
2025년부터 생긴 컬럼: {'일반_점포_수', '전체_점포_수'}


In [12]:
stores = pd.concat(store_list, ignore_index=True)
print(stores.shape)

print('분기 수:', stores['기준_년분기_코드'].nunique())
dup = stores.duplicated(['기준_년분기_코드', '상권_코드', '서비스_업종_코드'])
print('중복 행:', dup.sum())

(1680756, 16)
분기 수: 22
중복 행: 0


In [13]:
# 합친 뒤에는 옛 컬럼은 2025년부터, 새 컬럼은 2024년까지 비어 있어야 정상
stores['연도'] = stores['기준_년분기_코드'] // 10
print(stores.groupby('연도')[['점포_수', '유사_업종_점포_수', '전체_점포_수', '일반_점포_수']].count())
stores = stores.drop(columns='연도')

        점포_수  유사_업종_점포_수  전체_점포_수  일반_점포_수
연도                                        
2021  303880      303880        0        0
2022  305587      305587        0        0
2023  307741      307741        0        0
2024  306889      306889        0        0
2025       0           0   304775   304775
2026       0           0   151884   151884


## 4. 인구 3종, 집객시설, 영역

파일이 하나씩이라 바로 읽습니다. 상권 수가 파일마다 조금씩 다른데(영역 파일 1,650개 기준), 분석 테이블을 만들 때 LEFT JOIN으로 붙일 거라 여기서는 개수만 봐 둡니다.

In [14]:
floating = pd.read_csv(files['서울시 상권분석서비스(길단위인구-상권).csv'], encoding='cp949')
workers = pd.read_csv(files['서울시 상권분석서비스(직장인구-상권).csv'], encoding='cp949')
residents = pd.read_csv(files['서울시 상권분석서비스(상주인구-상권).csv'], encoding='cp949')
facilities = pd.read_csv(files['서울시 상권분석서비스(집객시설-상권).csv'], encoding='cp949')
areas = pd.read_csv(files['서울시_상권분석서비스(영역-상권).csv'], encoding='cp949')

for name, df in [('길단위인구', floating), ('직장인구', workers), ('상주인구', residents), ('집객시설', facilities)]:
    n_dup = df.duplicated(['기준_년분기_코드', '상권_코드']).sum()
    print(name, df.shape, '분기', df['기준_년분기_코드'].nunique(), '상권', df['상권_코드'].nunique(), '중복', n_dup)

print('영역', areas.shape, '상권', areas['상권_코드'].nunique())

길단위인구 (36281, 27) 분기 22 상권 1650 중복 0
직장인구 (36027, 26) 분기 22 상권 1643 중복 0
상주인구 (35908, 29) 분기 22 상권 1637 중복 0
집객시설 (34716, 25) 분기 22 상권 1578 중복 0
영역 (1650, 11) 상권 1650


In [15]:
# 집객시설은 빈 칸이 많음. 값이 있는 칸의 최솟값이 1이면 빈 칸 = 0개로 봐도 될 듯
fac_cols = facilities.columns[5:]
print(facilities[fac_cols].min())

집객시설_수      1.0
관공서_수       1.0
은행_수        1.0
종합병원_수      1.0
일반_병원_수     1.0
약국_수        1.0
유치원_수       1.0
초등학교_수      1.0
중학교_수       1.0
고등학교_수      1.0
대학교_수       1.0
백화점_수       1.0
슈퍼마켓_수      1.0
극장_수        1.0
숙박_시설_수     1.0
공항_수        4.0
철도_역_수      NaN
버스_터미널_수    1.0
지하철_역_수     1.0
버스_정거장_수    1.0
dtype: float64


In [16]:
areas['상권_구분_코드_명'].value_counts()

상권_구분_코드_명
골목상권    1090
전통시장     305
발달상권     249
관광특구       6
Name: count, dtype: int64

## 5. PostgreSQL에 적재

`if_exists='replace'`라서 다시 돌리면 테이블을 지우고 새로 만듭니다. 점포 테이블이 170만 행 정도라 몇 분 걸립니다.

In [17]:
tables = {
    '원본_추정매출': sales,
    '원본_점포': stores,
    '원본_길단위인구': floating,
    '원본_직장인구': workers,
    '원본_상주인구': residents,
    '원본_집객시설': facilities,
    '원본_영역': areas,
}

for table_name, df in tables.items():
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=5000, method='multi')
    print(table_name, len(df), '행 넣음')

원본_추정매출 481462 행 넣음


원본_점포 1680756 행 넣음


원본_길단위인구 36281 행 넣음


원본_직장인구 36027 행 넣음


원본_상주인구 35908 행 넣음


원본_집객시설 34716 행 넣음
원본_영역 1650 행 넣음


In [18]:
# DB에 들어간 행 수가 데이터프레임이랑 같은지 확인
with engine.connect() as conn:
    for table_name, df in tables.items():
        n = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}')).scalar()
        print(table_name, n, '같음' if n == len(df) else '다름!')

원본_추정매출 481462 같음


원본_점포 1680756 같음
원본_길단위인구 36281 같음
원본_직장인구 36027 같음
원본_상주인구 35908 같음
원본_집객시설 34716 같음
원본_영역 1650 같음


## 정리

- 7개 원본 테이블을 넣었습니다. 분기는 전부 2021년 1분기~2026년 2분기 22개이고 중복 행은 없습니다.
- 추정매출 시간대 컬럼 이름의 `~`를 `_`로 바꿨습니다. 값은 손대지 않았습니다.
- 점포 파일은 2025년부터 컬럼 구성이 달라서 그대로 합쳐 뒀습니다. 점포 수를 하나로 맞추는 건 `sql/01_분석_테이블.sql`에서 합니다.
- 다음: `02_전처리_검증.ipynb`에서 분석 테이블을 만들고 확인합니다.